## 1. <font color = red> Install and Import the Required Libraries

In [ ]:
# Install Required Libraries
!pip install -U -q pdfplumber chromadb google-genai pandas

# Import Required Libraries

import pdfplumber
from pathlib import Path
import pandas as pd
from operator import itemgetter
import json
import chromadb

# Google GenAI SDK
from google import genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 133.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.4/791.4 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 137.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. <font color = red> Read, Process, and Chunk the PDF Files

We will be using [pdfplumber](https://https://pypi.org/project/pdfplumber/) to read and process the PDF files.

`pdfplumber` allows for better parsing of the PDF file as it can read various elements of the PDF apart from the plain text, such as, tables, images, etc. It also offers wide functionaties and visual debugging features to help with advanced preprocessing as well.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


#### <font color = red> 2.2 Extracting text from multiple PDFs

Let's now try and read multiple documents, extract text from them using appropriate preprocessing, and store them in a dataframe


In [ ]:
# Define the path where all pdf documents are present
pdf_path = "/content/drive/MyDrive/PolicyMind_AI/PolicyDocuments/"

In [ ]:
# Function to check whether a word is present in a table or not for segregation of regular text and tables

def check_bboxes(word, table_bbox):
    # Check whether word is inside a table bbox.
    l = word['x0'], word['top'], word['x1'], word['bottom']
    r = table_bbox
    return l[0] > r[0] and l[1] > r[1] and l[2] < r[2] and l[3] < r[3]

In [ ]:
# Function to extract text from a PDF file.
# 1. Declare a variable p to store the iteration of the loop that will help us store page numbers alongside the text
# 2. Declare an empty list 'full_text' to store all the text files
# 3. Use pdfplumber to open the pdf pages one by one
# 4. Find the tables and their locations in the page
# 5. Extract the text from the tables in the variable 'tables'
# 6. Extract the regular words by calling the function check_bboxes() and checking whether words are present in the table or not
# 7. Use the cluster_objects utility to cluster non-table and table words together so that they retain the same chronology as in the original PDF
# 8. Declare an empty list 'lines' to store the page text
# 9. If a text element in present in the cluster, append it to 'lines', else if a table element is present, append the table
# 10. Append the page number and all lines to full_text, and increment 'p'
# 11. When the function has iterated over all pages, return the 'full_text' list

def extract_text_from_pdf(pdf_path):
    p = 0
    full_text = []


    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_no = f"Page {p+1}"
            text = page.extract_text()

            tables = page.find_tables()
            table_bboxes = [i.bbox for i in tables]
            tables = [{'table': i.extract(), 'top': i.bbox[1]} for i in tables]
            non_table_words = [word for word in page.extract_words() if not any(
                [check_bboxes(word, table_bbox) for table_bbox in table_bboxes])]
            lines = []

            for cluster in pdfplumber.utils.cluster_objects(non_table_words + tables, itemgetter('top'), tolerance=5):

                if 'text' in cluster[0]:
                    try:
                        lines.append(' '.join([i['text'] for i in cluster]))
                    except KeyError:
                        pass

                elif 'table' in cluster[0]:
                    lines.append(json.dumps(cluster[0]['table']))


            full_text.append([page_no, " ".join(lines)])
            p +=1

    return full_text

In [ ]:
# Define the directory containing the PDF files
pdf_directory = Path(pdf_path)

# Initialize an empty list to store the extracted texts and document names
data = []

# Loop through all files in the directory
for pdf_path in pdf_directory.glob("*.pdf"):

    # Process the PDF file
    print(f"...Processing {pdf_path.name}")

    # Call the function to extract the text from the PDF
    extracted_text = extract_text_from_pdf(pdf_path)

    # Convert the extracted list to a PDF, and add a column to store document names
    extracted_text_df = pd.DataFrame(extracted_text, columns=['Page No.', 'Page_Text'])
    extracted_text_df['Document Name'] = pdf_path.name

    # Append the extracted text and document name to the list
    data.append(extracted_text_df)

    # Print a message to indicate progress
    print(f"Finished processing {pdf_path.name}")

# Print a message to indicate all PDFs have been processed
print("All PDFs have been processed.")

...Processing HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf
Finished processing HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf
...Processing HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Finished processing HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
...Processing HDFC-Surgicare-Plan-101N043V01.pdf
Finished processing HDFC-Surgicare-Plan-101N043V01.pdf
...Processing HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf
Finished processing HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf
...Processing HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf
Finished processing HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf
...Processing HDFC-Life-Smart-Pension-Plan-Policy-Document-Online.pdf
Finished processing HDFC-Life-Smart-Pension-Plan-Policy-Document-Online.pdf
...Processing HDFC-Life-Group-Term-Life-Policy.pdf
Finished processing HDFC-Life-Group-T

In [ ]:
# Concatenate all the DFs in the list 'data' together

insurance_pdfs_data = pd.concat(data, ignore_index=True)

In [ ]:
insurance_pdfs_data.head()

,Page No.,Page_Text,Document Name
0,Page 1,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...
1,Page 2,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...
2,Page 3,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...
3,Page 4,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...
4,Page 5,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...


In [ ]:
# Check one of the extracted page texts to ensure that the text has been correctly read

insurance_pdfs_data.Page_Text[2]

'HDFC Life Sanchay Plus (UIN – 101N134V19) – Appendix 9 (c) – Policy Bond A non-participating, non-linked savings insurance plan POLICY DOCUMENT- HDFC Life Sanchay Plus Unique Identification Number: <<101N134V19>> Your Policy is a non-participating non-linked savings insurance policy. This document is the evidence of a contract between HDFC Life Insurance Company Limited and the Policyholder as described in the Policy Schedule given below. This Policy is based on the proposal made by the within named Policyholder and submitted to the Company along with the required documents, declarations, statements, any response given to medical questionnaire by the Life Assured, applicable medical evidence and other information received by the Company from the Policyholder, Life Assured or on behalf of the Policyholder (“Proposal”). This Policy is effective upon receipt and realisation, by the Company, of the consideration payable as First Premium under the Policy. This Policy is written under and w

In [ ]:
# Let's also check the length of all the texts as there might be some empty pages or pages with very few words that we can drop

insurance_pdfs_data['Text_Length'] = insurance_pdfs_data['Page_Text'].apply(lambda x: len(x.split(' ')))

In [ ]:
insurance_pdfs_data['Text_Length']

0      552
1      137
2      342
3      244
4       46
      ... 
212     51
213    255
214    691
215    648
216    160
Name: Text_Length, Length: 217, dtype: int64

In [ ]:
# Retain only the rows with a text length of at least 10

insurance_pdfs_data = insurance_pdfs_data.loc[insurance_pdfs_data['Text_Length'] >= 10]
insurance_pdfs_data

,Page No.,Page_Text,Document Name,Text_Length
0,Page 1,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,552
1,Page 2,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,137
2,Page 3,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,342
3,Page 4,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,244
4,Page 5,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,46
...,...,...,...,...
212,Page 26,The Council for Insurance Ombudsmen shall deve...,HDFC-Life-Group-Term-Life-Policy.pdf,51
213,Page 27,ANNEXURE - A – COVERAGE SCHEDULE (Forming part...,HDFC-Life-Group-Term-Life-Policy.pdf,255
214,Page 28,ANNEXURE - B Section 45 – Policy shall not be ...,HDFC-Life-Group-Term-Life-Policy.pdf,691
215,Page 29,ANNEXURE - C Section 39 – Nomination by Policy...,HDFC-Life-Group-Term-Life-Policy.pdf,648


In [ ]:
# Store the metadata for each page in a separate column

insurance_pdfs_data['Metadata'] = insurance_pdfs_data.apply(lambda x: {'Policy_Name': x['Document Name'][:-4], 'Page_No.': x['Page No.']}, axis=1)

In [ ]:
insurance_pdfs_data.head()

,Page No.,Page_Text,Document Name,Text_Length,Metadata
0,Page 1,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,552,{'Policy_Name': 'HDFC-Life-Sanchay-Plus-Life-L...
1,Page 2,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,137,{'Policy_Name': 'HDFC-Life-Sanchay-Plus-Life-L...
2,Page 3,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,342,{'Policy_Name': 'HDFC-Life-Sanchay-Plus-Life-L...
3,Page 4,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,244,{'Policy_Name': 'HDFC-Life-Sanchay-Plus-Life-L...
4,Page 5,HDFC Life Sanchay Plus (UIN – 101N134V19) – Ap...,HDFC-Life-Sanchay-Plus-Life-Long-Income-Option...,46,{'Policy_Name': 'HDFC-Life-Sanchay-Plus-Life-L...


## 3. <font color = red> Generate and Store Embeddings using Gemini Embeddings and ChromaDB

In this section, we generate embeddings for the chunked insurance policy documents using Google's `gemini-embedding-001` model and store them in a ChromaDB collection for semantic retrieval in the RAG pipeline.

In [ ]:
# Set Google API Key for Gemma 4 and Gemini Embeddings

filepath = "/content/drive/MyDrive/PolicyMind_AI/"

with open(filepath + "Google_API_Key.txt", "r") as f:

    GOOGLE_API_KEY = ''.join(f.readlines()).strip()

In [ ]:
# Initialize Google GenAI Client

from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

In [ ]:
# Define path for storing ChromaDB collections

chroma_data_path = "/content/drive/MyDrive/PolicyMind_AI/chromaDB"

# Import ChromaDB

import chromadb

# Initialize Persistent Chroma Client

chroma_client = chromadb.PersistentClient(
    path=chroma_data_path
)

In [ ]:
# Define Gemini Embedding Model

embedding_model = "gemini-embedding-001"

In [ ]:
# Initialize ChromaDB Collection for PolicyMind_AI

insurance_collection = chroma_client.get_or_create_collection(
    name="PolicyMind_AI"
)

In [ ]:
# Convert insurance policy text and metadata into lists

documents_list = insurance_pdfs_data["Page_Text"].tolist()

metadata_list = insurance_pdfs_data["Metadata"].tolist()

In [ ]:
# Add insurance policy documents and metadata to ChromaDB collection

insurance_collection.add(

    documents=documents_list,

    ids=[str(i) for i in range(0, len(documents_list))],

    metadatas=metadata_list
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 100MiB/s] 


In [ ]:
# View first few entries from the ChromaDB collection

insurance_collection.get(

    ids=['0', '1', '2'],

    include=['documents', 'metadatas']
)

{'ids': ['0', '1', '2'],
 'embeddings': None,
 'documents': ['HDFC Life Sanchay Plus (UIN – 101N134V19) – Appendix 9 (c) – Policy Bond A non-participating, non-linked savings insurance plan Part A <<01 January 2021>> <<Policyholder’s Name>> <<Policyholder’s Address>> <<Policyholder’s Contact Number>> Dear <<Policyholder’s Name>>, Sub: Your Policy no. <<>> We are glad to inform you that your proposal has been accepted and the HDFC Life Sanchay Plus Policy (“Policy”) being this Policy, has been issued. We have made every effort to design your Policy in a simple format. We have highlighted items of importance so that you may recognise them easily. Policy document: As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy is enclosed herewith. Please preserve this document safely and also inform your Nominees about the same. A copy of your proposal form submitted by you is enclosed for your information and record. Cancellation in the Free-Look

## 4. <font color = red> Semantic Search with Cache

In this section, we will perform a semantic search of a query in the collections embeddings to get several top semantically similar results.

In [ ]:
# Create cache collection for storing previous insurance query responses

cache_collection = chroma_client.get_or_create_collection(
    name="Insurance_Cache"
)

In [ ]:
cache_collection.peek()

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': []}

In [ ]:
# Read user insurance-related query

query = input("Ask your insurance policy question: ")

Ask your insurance policy question: Can I claim multiple surgeries under this policy?


In [ ]:
# Search the cache collection first

cache_results = cache_collection.query(

    query_texts=[query],

    n_results=1
)

In [ ]:
# cache_results

In [ ]:
# Query the insurance policy collection

results = insurance_collection.query(

    query_texts=[query],

    n_results=10
)

# View retrieved results
# results.items()

In [ ]:
# Implement Cache-Based Semantic Search

threshold = 0.2

ids = []
documents = []
distances = []
metadatas = []

results_df = pd.DataFrame()

# Check cache similarity distance

if cache_results['distances'][0] == [] or cache_results['distances'][0][0] > threshold:

    # Query main insurance collection

    results = insurance_collection.query(

        query_texts=[query],

        n_results=10
    )

    # Store query and retrieval results in cache

    Keys = []
    Values = []

    for key, val in results.items():

        if val is None:
            continue

        for i in range(10):

            try:

                if isinstance(val[0], list):
                    Values.append(str(val[0][i]))

                else:
                    Values.append(str(val[i]))

            except (IndexError, TypeError):

                break

            Keys.append(str(key) + str(i))

    # Add query to cache collection

    cache_collection.add(

        documents=[query],

        ids=[query],

        metadatas=[dict(zip(Keys, Values))]
    )

    print("Not found in cache. Retrieved from main insurance collection.")

    result_dict = {

        "Metadatas": results['metadatas'][0],

        "Documents": results['documents'][0],

        "Distances": results['distances'][0],

        "IDs": results['ids'][0]
    }

    results_df = pd.DataFrame.from_dict(result_dict)

    display(results_df)


# If found in cache

elif cache_results['distances'][0][0] <= threshold:

    cache_result_dict = cache_results['metadatas'][0][0]

    for key, value in cache_result_dict.items():

        if 'ids' in key:
            ids.append(value)

        elif 'documents' in key:
            documents.append(value)

        elif 'distances' in key:
            distances.append(value)

        elif 'metadatas' in key:
            metadatas.append(value)

    print("Found in cache!")

    results_df = pd.DataFrame({

        "IDs": ids,

        "Documents": documents,

        "Distances": distances,

        "Metadatas": metadatas
    })

    display(results_df)

Not found in cache. Retrieved from main insurance collection.


,Metadatas,Documents,Distances,IDs
0,"{'Page_No.': 'Page 10', 'Policy_Name': 'HDFC-S...",HDFC Standard Life Insurance Company Limited H...,0.829261,68
1,"{'Page_No.': 'Page 13', 'Policy_Name': 'HDFC-L...",Part D 1. Claims Procedure You have the option...,0.898589,38
2,{'Policy_Name': 'HDFC-Surgicare-Plan-101N043V0...,HDFC Standard Life Insurance Company Limited H...,0.902452,65
3,{'Policy_Name': 'HDFC-Surgicare-Plan-101N043V0...,HDFC Standard Life Insurance Company Limited H...,0.910736,71
4,"{'Page_No.': 'Page 16', 'Policy_Name': 'HDFC-L...",Part F 1. Waiting Period  60 days waiting per...,0.957397,41
5,"{'Page_No.': 'Page 11', 'Policy_Name': 'HDFC-S...",HDFC Standard Life Insurance Company Limited H...,0.983846,69
6,"{'Page_No.': 'Page 26', 'Policy_Name': 'HDFC-L...",Annexure I LIST OF 138 SURGERIES  The Surgeri...,0.988116,51
7,{'Policy_Name': 'HDFC-Life-Easy-Health-101N110...,v. In case 100% of the Sum Insured has been us...,1.026670,37
8,{'Policy_Name': 'HDFC-Life-Group-Poorna-Suraks...,"[[""21. Progressive\nScleroderma"", ""22. Muscula...",1.053903,119
9,"{'Page_No.': 'Page 15', 'Policy_Name': 'HDFC-S...",HDFC Standard Life Insurance Company Limited H...,1.067947,73


In [ ]:
# Sort semantic results by distance score

top_3_semantic = results_df.sort_values(
    by="Distances",
    ascending=True
)

# Select top 3 most relevant chunks

top_3_RAG = top_3_semantic[
    ["Documents", "Metadatas"]
][:10]

## 6. Retrieval Augmented Generation (RAG)

Now that we have retrieved the most relevant document chunks using semantic search, we pass the user query along with the retrieved insurance policy chunks to the Gemma 4 model using a carefully engineered prompt. The model then generates a grounded, user-friendly response with relevant policy citations instead of returning raw document chunks directly.

In [ ]:

# Function to generate response using Gemma 4

def generate_response(query, top_3_RAG):

    """
    Generate insurance policy response using Gemma 4
    based on retrieved insurance document chunks.
    """

    prompt = f"""

    You are an intelligent AI assistant specialized in insurance policy analysis.

    A user has asked the following insurance-related question:

    USER QUESTION:
    {query}


    You are also provided with retrieved policy document content:

    RETRIEVED DOCUMENTS:
    {top_3_RAG}


    Instructions:

    1. Answer ONLY using the provided insurance document content.

    2. Do not hallucinate or generate unsupported policy information.

    3. If the answer is partially available, clearly mention limitations.

    4. If relevant information exists in tables, summarize it clearly.

    5. Mention policy names and page numbers from metadata as citations.

    6. Keep the response professional, concise, and customer-friendly.

    7. If the query is unrelated to insurance documents, say:
       'The query is not relevant to the available insurance policy documents.'

    8. Provide the final answer first.

    9. Then provide citations separately.


    RESPONSE FORMAT:

    Answer:
    <generated answer>

    Citations:
    - Policy Name | Page Number
    """


    response = client.models.generate_content(

        model="gemma-4-26b-a4b-it",

        contents=prompt
    )

    return response.text

In [ ]:
# Generate response from Gemma 4

response = generate_response(query, top_3_RAG)

# Print final response

print(response)

Answer:
The provided documents do not explicitly state whether multiple surgeries can be claimed. However, they indicate that claims are subject to the Sum Insured, noting a provision for the case where "100% of the Sum Insured has been used." Please note that the documents provided are snippets from various policies and do not contain a definitive rule regarding the number of surgeries allowed.

Citations:
- HDFC-Life-Easy-Health-101N110... | Not specified
